<a href="https://colab.research.google.com/github/rajaveni08-rgb/Trendpulse_Rajaveni/blob/main/MiniProject_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import time
import json
import requests
from datetime import datetime

def collect_hn_trends():
    # Define the official API endpoints
    TOP_STORIES_URL = "https://hacker-news.firebaseio.com/v0/topstories.json"
    ITEM_DETAIL_URL = "https://hacker-news.firebaseio.com/v0/item/{id}.json"

    # 1 — Make the API Calls: Establish headers
    headers = {"User-Agent": "TrendPulse/1.0"}

    # Define exact category keyword mappings (all converted down to lowercase for strict comparison)
    category_map = {
        "technology": ["ai", "software", "tech", "code", "computer", "data", "cloud", "api", "gpu", "llm"],
        "worldnews": ["war", "government", "country", "president", "election", "climate", "attack", "global"],
        "sports": ["nfl", "nba", "fifa", "sport", "game", "team", "player", "league", "championship"],
        "science": ["research", "study", "space", "physics", "biology", "discovery", "nasa", "genome"],
        "entertainment": ["movie", "film", "music", "netflix", "game", "book", "show", "award", "streaming"]
    }

    # Track collected payloads and separate counts per group
    all_collected_stories = []
    category_counts = {cat: 0 for cat in category_map}

    print("Fetching top 500 story IDs from HackerNews...")
    try:
        response = requests.get(TOP_STORIES_URL, headers=headers, timeout=10)
        response.raise_for_status()
        story_ids = response.json()[:500] # Restrict array to first 500 items
    except Exception as e:
        print(f"Failed to fetch top stories index array: {e}")
        return

    # Loop through each category sequentially to process its respective batch
    for category, keywords in category_map.items():
        print(f"Scanning for category: {category}...")

        for story_id in story_ids:
            # 2 — Extract the Fields: Limit collection to 25 items maximum per category bucket
            if category_counts[category] >= 25:
                break

            try:
                # Construct query endpoint URL dynamically
                url = ITEM_DETAIL_URL.format(id=story_id)
                item_response = requests.get(url, headers=headers, timeout=5)

                # If a request fails, print a message and move on safely — don't crash
                if item_response.status_code != 200:
                    print(f"Skipping ID {story_id}: HTTP status code {item_response.status_code}")
                    continue

                story = item_response.json()

                # Verify valid story item container object
                if not story or "title" not in story:
                    continue

                title_lower = story["title"].lower()

                # Check for case-insensitive matching conditions
                if any(kw in title_lower for kw in keywords):
                    # Structure fields to exact requested schema requirements
                    story_payload = {
                        "post_id": story.get("id"),
                        "title": story.get("title"),
                        "category": category,
                        "score": story.get("score", 0),
                        "num_comments": story.get("descendants", 0),  # Correctly map descendants property
                        "author": story.get("by", "unknown"),
                        "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    }

                    all_collected_stories.append(story_payload)
                    category_counts[category] += 1

            except Exception as inner_err:
                # Catch dynamic inner request exceptions to guarantee pipeline loop continuity
                print(f"Skipping unreadable item ID {story_id}: {inner_err}")
                continue

        print(f"Finished {category}. Total gathered for group: {category_counts[category]}")
        # Wait 2 seconds between each category loop execution block
        print("Enforcing mandatory cool-down interval...")
        time.sleep(2)

    # 3 — Save to a JSON File
    os.makedirs("data", exist_ok=True)
    current_date = datetime.now().strftime("%Y%m%d")
    output_path = f"data/trends_{current_date}.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(all_collected_stories, f, indent=4, ensure_ascii=False)

    # Output final expected console execution confirmation log string
    print(f"Collected {len(all_collected_stories)} stories. Saved to {output_path}")

if __name__ == "__main__":
    collect_hn_trends()


Fetching top 500 story IDs from HackerNews...
Scanning for category: technology...
Finished technology. Total gathered for group: 25
Enforcing mandatory cool-down interval...
Scanning for category: worldnews...
Finished worldnews. Total gathered for group: 24
Enforcing mandatory cool-down interval...
Scanning for category: sports...
Finished sports. Total gathered for group: 8
Enforcing mandatory cool-down interval...
Scanning for category: science...
Finished science. Total gathered for group: 10
Enforcing mandatory cool-down interval...
Scanning for category: entertainment...
Finished entertainment. Total gathered for group: 25
Enforcing mandatory cool-down interval...
Collected 92 stories. Saved to data/trends_20260929.json


In [2]:
import os
import glob
import pandas as pd

def process_trend_data():
    # Target path for output
    CLEAN_CSV_PATH = "data/trends_clean.csv"

    print("Step 1: Locating latest JSON dataset from Task 1...")
    # Find all trend JSON files inside the data folder
    json_files = glob.glob("data/trends_*.json")

    if not json_files:
        raise FileNotFoundError("Error: No raw trend JSON files found in data/. Please run Task 1 first.")

    # Identify the most recently created JSON file to process the fresh data pipeline
    latest_json_file = max(json_files, key=os.path.getctime)
    print(f"Loading data from file: {latest_json_file}")

    # Read the JSON file into a pandas DataFrame
    try:
        df = pd.read_json(latest_json_file)
    except Exception as e:
        print(f"Error reading JSON file contents: {e}")
        return

    print(f"Initial raw rows loaded: {len(df)}")

    print("\nStep 2: Cleaning and standardizing fields...")

    # 1. Handle missing values: drop records missing critical core identifiers
    df.dropna(subset=["post_id", "title"], inplace=True)

    # 2. String cleaning: Strip leading/trailing whitespaces from strings to avoid parsing discrepancies
    df["title"] = df["title"].astype(str).str.strip()
    df["author"] = df["author"].astype(str).str.strip()
    df["category"] = df["category"].astype(str).str.strip().str.lower()

    # 3. Numeric alignment: Coerce metrics into valid integers and fill NaN markers with 0
    df["score"] = pd.to_numeric(df["score"], errors="coerce").fillna(0).astype(int)
    df["num_comments"] = pd.to_numeric(df["num_comments"], errors="coerce").fillna(0).astype(int)

    # 4. Remove exact duplicate postings across overlapping keywords if a story appeared in multiple passes
    initial_count = len(df)
    df.drop_duplicates(subset=["post_id"], keep="first", inplace=True)
    dedup_count = initial_count - len(df)
    if dedup_count > 0:
        print(f"Removed {dedup_count} duplicate cross-category postings.")

    print("\nStep 3: Exporting clean tabular matrix to CSV...")
    # Ensure destination directory structure exists safely
    os.makedirs("data", exist_ok=True)

    # Save structured table down to file (excluding pandas index tracking)
    df.to_csv(CLEAN_CSV_PATH, index=False, encoding="utf-8")

    print("=" * 50)
    print(f"Pipeline processing complete. Cleaned data saved to: {CLEAN_CSV_PATH}")
    print(f"Final data volume: {len(df)} standardized records available.")
    print("=" * 50)

if __name__ == "__main__":
    process_trend_data()


Step 1: Locating latest JSON dataset from Task 1...
Loading data from file: data/trends_20260929.json
Initial raw rows loaded: 92

Step 2: Cleaning and standardizing fields...
Removed 11 duplicate cross-category postings.

Step 3: Exporting clean tabular matrix to CSV...
Pipeline processing complete. Cleaned data saved to: data/trends_clean.csv
Final data volume: 81 standardized records available.


In [3]:
import os
import pandas as pd

def analyze_trend_metrics():
    # Input and output file paths
    INPUT_CSV_PATH = "data/trends_clean.csv"
    OUTPUT_CSV_PATH = "data/trends_analysed.csv"

    print("Step 1: Ingesting processed dataset...")
    # Verify the target raw clean data table file exists before parsing
    if not os.path.exists(INPUT_CSV_PATH):
        raise FileNotFoundError(f"Error: Missing clean source file: {INPUT_CSV_PATH}. Please execute Task 2 first.")

    df = pd.read_csv(INPUT_CSV_PATH)
    print(f"Loaded {len(df)} records for statistical processing.")

    print("\nStep 2: Aggregating structural baseline metrics...")
    # Calculate a baseline metric representing total user engagement volume
    total_network_score = df["score"].sum()
    print(f"Total community engagement score across dataset: {total_network_score}")

    # 1. Feature Engineering: Compute engagement share percent for every story
    # Handles division by zero gracefully using conditional default matching if dataset is empty
    if total_network_score > 0:
        df["score_share_pct"] = (df["score"] / total_network_score) * 100
    else:
        df["score_share_pct"] = 0.0

    # Round fractional outputs cleanly to 2 decimal places
    df["score_share_pct"] = df["score_share_pct"].round(2)

    # 2. Advanced Feature Engineering: Create a calculated 'Pulse Score' formula.
    # Formula weighing model: Score (upvotes) weighted at 60%, Comments weighted at 40%
    df["pulse_score"] = (df["score"] * 0.6) + (df["num_comments"] * 0.4)
    df["pulse_score"] = df["pulse_score"].round(2)

    print("\nStep 3: Calculating group analytics summaries...")
    # Analyze and display mean and total values grouped by category matrix fields
    category_summary = df.groupby("category").agg(
        total_stories=("post_id", "count"),
        average_score=("score", "mean"),
        total_comments=("num_comments", "sum"),
        average_pulse=("pulse_score", "mean")
    ).round(2)

    print("\n--- Summary Performance Matrix by Category ---")
    print(category_summary)
    print("----------------------------------------------")

    print("\nStep 4: Sorting datasets based on calculated structural momentum...")
    # Rank all items globally using your engineered dynamic performance index
    df.sort_values(by="pulse_score", ascending=False, inplace=True)

    # Create an explicit index numbering row alignment following sorting operations
    df.reset_index(drop=True, inplace=True)

    print("\nStep 5: Exporting processed telemetry layer...")
    # Save the complete analyzed data package out to disk
    df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8")

    print("=" * 50)
    print(f"Analysis engine execution complete.")
    print(f"Augmented metrics table saved safely to: {OUTPUT_CSV_PATH}")
    print("=" * 50)

if __name__ == "__main__":
    analyze_trend_metrics()


Step 1: Ingesting processed dataset...
Loaded 81 records for statistical processing.

Step 2: Aggregating structural baseline metrics...
Total community engagement score across dataset: 13133

Step 3: Calculating group analytics summaries...

--- Summary Performance Matrix by Category ---
               total_stories  average_score  total_comments  average_pulse
category                                                                  
entertainment             18         180.17            1601         143.68
science                   10         128.20            1205         125.12
sports                     6          58.00             188          47.33
technology                25         168.88            2584         142.67
worldnews                 22         183.55            2668         158.64
----------------------------------------------

Step 4: Sorting datasets based on calculated structural momentum...

Step 5: Exporting processed telemetry layer...
Analysis engine execu

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt

def generate_trend_visualizations():
    # Input data location and output paths configuration
    INPUT_CSV_PATH = "data/trends_analysed.csv"
    OUTPUT_DIR = "outputs"

    print("Step 1: Ingesting analyzed data layers...")
    # Verify input exists before plotting elements
    if not os.path.exists(INPUT_CSV_PATH):
        raise FileNotFoundError(f"Error: Missing analyzed source file: {INPUT_CSV_PATH}. Run Task 3 first.")

    df = pd.read_csv(INPUT_CSV_PATH)
    print(f"Loaded {len(df)} records for chart layout generation.")

    # Establish outputs directory structure if missing
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Apply modern baseline visual design styles for cleaner rendering outputs
    plt.style.use('ggplot')

    # ----------------------------------------------------
    # 1. Chart 1: Horizontal Bar Chart of Top Stories
    # ----------------------------------------------------
    print("Generating Chart 1: Top Stories Rankings...")
    plt.figure(figsize=(10, 5))
    top_5 = df.head(5)

    # Invert order for barh alignment matching highest value on top
    plt.barh(top_5['title'], top_5['pulse_score'], color='royalblue', edgecolor='black')
    plt.gca().invert_yaxis()

    plt.title('Top 5 Viral HackerNews Stories (by Pulse Score)', fontsize=12, fontweight='bold')
    plt.xlabel('Calculated Pulse Score Matrix Index')
    plt.ylabel('Story Headline Title')
    plt.tight_layout()

    chart1_path = os.path.join(OUTPUT_DIR, "chart1_top_stories.png")
    plt.savefig(chart1_path, dpi=150)
    plt.close()

    # ----------------------------------------------------
    # 2. Chart 2: Total Volume by Category
    # ----------------------------------------------------
    print("Generating Chart 2: Category Breakdown...")
    plt.figure(figsize=(8, 5))

    category_volume = df.groupby('category')['score'].sum().sort_values(ascending=False)
    category_volume.plot(kind='bar', color='seagreen', edgecolor='black')

    plt.title('Total Engagement Score Distributed by Category', fontsize=12, fontweight='bold')
    plt.xlabel('Category Filter Group')
    plt.ylabel('Aggregate Upvote Scores')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()

    chart2_path = os.path.join(OUTPUT_DIR, "chart2_categories.png")
    plt.savefig(chart2_path, dpi=150)
    plt.close()

    # ----------------------------------------------------
    # 3. Chart 3: Scatter Plot (Score Share vs Pulse Score)
    # ----------------------------------------------------
    print("Generating Chart 3: Metric Momentum Scatter Map...")
    plt.figure(figsize=(8, 6))

    plt.scatter(df['score_share_pct'], df['pulse_score'], s=120, color='crimson', alpha=0.7, edgecolors='black')

    # Label top 3 points explicitly on the scatter mapping
    for i in range(min(3, len(df))):
        plt.annotate(
            df['title'].iloc[i][:30] + "...",
            (df['score_share_pct'].iloc[i], df['pulse_score'].iloc[i]),
            fontsize=8,
            xytext=(5, 5),
            textcoords='offset points'
        )

    plt.title('Trend Momentum Matrix: Engagement Share vs. Pulse Score', fontsize=12, fontweight='bold')
    plt.xlabel('Global Score Share Percentage (%)')
    plt.ylabel('Composite Pulse Score')
    plt.tight_layout()

    chart3_path = os.path.join(OUTPUT_DIR, "chart3_scatter.png")
    plt.savefig(chart3_path, dpi=150)
    plt.close()

    # ----------------------------------------------------
    # 4. BONUS Dashboard: 2x2 Combined Visual Layout Matrix
    # ----------------------------------------------------
    print("Assembling Bonus Asset: Consolidated Executive Analytics Dashboard...")
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('TrendPulse Pipeline Summary Executive Dashboard', fontsize=16, fontweight='bold', color='#333333')

    # Subplot: Horizontal Bar
    axs[0, 0].barh(top_5['title'].str.slice(0, 35) + "...", top_5['pulse_score'], color='royalblue')
    axs[0, 0].invert_yaxis()
    axs[0, 0].set_title('Top Volume Rankings Index', fontsize=10, fontweight='bold')
    axs[0, 0].set_xlabel('Pulse Score')

    # Subplot: Category Volume
    category_volume.plot(kind='bar', ax=axs[0, 1], color='seagreen')
    axs[0, 1].set_title('Total Category Share Distribution', fontsize=10, fontweight='bold')
    axs[0, 1].set_ylabel('Total Upvotes')
    axs[0, 1].tick_params(axis='x', rotation=30)

    # Subplot: Scatter Matrix Map
    axs[1, 0].scatter(df['score_share_pct'], df['pulse_score'], s=60, color='crimson', alpha=0.6)
    axs[1, 0].set_title('Momentum Cluster Quadrant Map', fontsize=10, fontweight='bold')
    axs[1, 0].set_xlabel('Score Share (%)')
    axs[1, 0].set_ylabel('Pulse Score')

    # Subplot: Descriptive Metadata Summary Table
    axs[1, 1].axis('off') # Clean canvas slate
    summary_data = df[['category', 'score', 'num_comments']].groupby('category').mean().round(1).reset_index()
    summary_data.columns = ['Category', 'Avg Upvotes', 'Avg Comments']

    table = axs[1, 1].table(
        cellText=summary_data.values,
        colLabels=summary_data.columns,
        loc='center',
        cellLoc='center',
        colColours=['#dddddd']*3
    )
    table.scale(1.0, 1.8)
    table.set_fontsize(10)
    axs[1, 1].set_title('Category Operational Performance Summary', fontsize=10, fontweight='bold')

    # Optimize dynamic layouts to keep labels from wrapping or clipping over margins
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    dashboard_path = os.path.join(OUTPUT_DIR, "dashboard.png")
    plt.savefig(dashboard_path, dpi=200)
    plt.close()

    print("=" * 50)
    print("Visualization processing cycle complete.")
    print(f"All assets verified and rendered inside corporate folder: /{OUTPUT_DIR}")
    print("Files ready for submission checklist review.")
    print("=" * 50)

if __name__ == "__main__":
    generate_trend_visualizations()


Step 1: Ingesting analyzed data layers...
Loaded 81 records for chart layout generation.
Generating Chart 1: Top Stories Rankings...
Generating Chart 2: Category Breakdown...
Generating Chart 3: Metric Momentum Scatter Map...
Assembling Bonus Asset: Consolidated Executive Analytics Dashboard...
Visualization processing cycle complete.
All assets verified and rendered inside corporate folder: /outputs
Files ready for submission checklist review.


In [24]:
### Project Directory Structure

```text
TrendPulse/
│
├── data/
│   ├── trends_20260929.json       # Generated dynamic collection snapshot
│   ├── trends_clean.csv           # Clean structured base table
│   └── trends_analysed.csv        # Advanced mathematical summary layer
│
├── outputs/
│   ├── chart1_top_stories.png     # Top 5 stories ranking graph
│   ├── chart2_categories.png      # Category distribution breakdown plot
│   ├── chart3_scatter.png         # Cluster momentum scatter chart
│   └── dashboard.png              # Combined 2x2 executive matrix layout
│
├── task1_data_collection.py       # API connection & categorization logic
├── task2_data_processing.py       # Data sanitization and standardizing script
├── task3_analysis.py              # Performance indicator logic execution
├── task4_visualization.py         # Matplotlib presentation generator
└── run_pipeline_tests.py          # Automated testing verification script
```

SyntaxError: invalid character '│' (U+2502) (606470411.py, line 5)

In [14]:
!ls -F

data/	      task1_data_collection.py	task4_visualization.py
outputs/      task2_data_processing.py
sample_data/  task3_analysis.py
